# 从原始文本，到模型能处理的张量

## 核心主线：一条文本的“变形记”

想象我们有一本小说，要把它变成训练 GPT 模型的数据。整个过程就像一条流水线，数据在每一步都会“变形”。

## 全流程概览图

## 第一步：原始文本 → 词块列表（分词）

In [2]:
import re

raw_text = "I HAD always thought Jack Gisburn rather a cheap genius"

# 正则表达式：按空格、标点来切分
tokens = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
tokens = [t.strip() for t in tokens if t.strip()]

print("分词结果:", tokens)
print("词块数量:", len(tokens))

分词结果: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius']
词块数量: 10


变形的意义：从一整块字符串，变成了一个个独立的语义单元。模型不再看“一整句话”，而是看“一个词一个词”。

## 第二步：词块列表 → 整数ID列表（编码）

In [3]:
# 建立词表（实际GPT-2词表有50257个词）
all_tokens = sorted(set(tokens))
vocab = {token: i for i, token in enumerate(all_tokens)}

print("词表（词→ID）:")
for word, idx in vocab.items():
    print(f"  '{word}' → {idx}")

# 编码
ids = [vocab[t] for t in tokens]
print(f"\n编码结果: {ids}")

词表（词→ID）:
  'Gisburn' → 0
  'HAD' → 1
  'I' → 2
  'Jack' → 3
  'a' → 4
  'always' → 5
  'cheap' → 6
  'genius' → 7
  'rather' → 8
  'thought' → 9

编码结果: [2, 1, 5, 9, 3, 0, 8, 4, 6, 7]


## 第三步：整数序列 → (输入, 目标) 对（数据采样）

In [5]:
import torch

# 假设我们把整本小说编码后，得到这个长序列（这里只演示一小段）
all_ids = [2, 1, 5, 9, 3, 0, 8, 4, 6, 7]  # 实际有5000+个

# 超参数
max_length = 4   # 上下文窗口：每次看4个词
stride = 2       # 步长：每次窗口移动2个位置

print(f"编码后的整数序列（共{len(all_ids)}个）:\n{all_ids}\n")

# 滑动窗口生成训练样本
input_chunks = []
target_chunks = []

for i in range(0, len(all_ids) - max_length, stride):
    input_chunk = all_ids[i : i + max_length]         # 取4个作为输入
    target_chunk = all_ids[i + 1 : i + max_length + 1] # 输入右移一位作为目标
    input_chunks.append(input_chunk)
    target_chunks.append(target_chunk)
    print(f"窗口{i//stride}: 输入={input_chunk} → 目标={target_chunk}")

print(input_chunks)
print(target_chunks)


编码后的整数序列（共10个）:
[2, 1, 5, 9, 3, 0, 8, 4, 6, 7]

窗口0: 输入=[2, 1, 5, 9] → 目标=[1, 5, 9, 3]
窗口1: 输入=[5, 9, 3, 0] → 目标=[9, 3, 0, 8]
窗口2: 输入=[3, 0, 8, 4] → 目标=[0, 8, 4, 6]
[[2, 1, 5, 9], [5, 9, 3, 0], [3, 0, 8, 4]]
[[1, 5, 9, 3], [9, 3, 0, 8], [0, 8, 4, 6]]


In [7]:
# 转成PyTorch张量
inputs = torch.tensor(input_chunks)   # shape: (样本数, 上下文长度)
targets = torch.tensor(target_chunks) # shape: (样本数, 上下文长度
print("\n最终训练数据:")
print(f"输入张量形状: {inputs.shape} ← (样本数=3, 上下文长度=4)")
print(f"输入张量:\n{inputs}")
print(f"\n目标张量形状: {targets.shape}")
print(f"目标张量:\n{targets}")


最终训练数据:
输入张量形状: torch.Size([3, 4]) ← (样本数=3, 上下文长度=4)
输入张量:
tensor([[2, 1, 5, 9],
        [5, 9, 3, 0],
        [3, 0, 8, 4]])

目标张量形状: torch.Size([3, 4])
目标张量:
tensor([[1, 5, 9, 3],
        [9, 3, 0, 8],
        [0, 8, 4, 6]])


## 第四步：整数ID → 稠密向量（词嵌入）

In [8]:
# 参数设置
vocab_size = 10       # 假设词表有10个词（实际GPT-2是50257）
embedding_dim = 256   # 每个词用256维向量表示（GPT-3用12288维）

# 创建嵌入层（这就是一个可学习的查询表）
token_embedding_layer = torch.nn.Embedding(vocab_size, embedding_dim)
print(f"嵌入层权重形状: {token_embedding_layer.weight.shape}")
print(f"解读: {vocab_size}个词，每个词对应一个{embedding_dim}维向量")

# 查表：把输入的整数ID转成向量
# inputs形状: (3, 4) → 3个样本，每个4个词
token_embeddings = token_embedding_layer(inputs)
print(f"\n输入形状: {inputs.shape}")
print(f"输出形状: {token_embeddings.shape}")
print(f"解读: {inputs.shape[0]}个样本, 每个{inputs.shape[1]}个词, 每个词变成{embedding_dim}维向量")

嵌入层权重形状: torch.Size([10, 256])
解读: 10个词，每个词对应一个256维向量

输入形状: torch.Size([3, 4])
输出形状: torch.Size([3, 4, 256])
解读: 3个样本, 每个4个词, 每个词变成256维向量


## 第五步：注入位置信息（位置嵌入）

In [10]:
# 创建位置嵌入层
context_length = 4  # 一共4个位置
pos_embedding_layer = torch.nn.Embedding(context_length, embedding_dim)

# 获取位置0,1,2,3的向量
# torch.arange(4) = [0, 1, 2, 3]
# 相当于说：“给我位置0、1、2、3的专属向量”
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(f"位置嵌入形状: {pos_embeddings.shape}")
print(f"解读: {context_length}个位置，每个位置是{embedding_dim}维向量")

位置嵌入形状: torch.Size([4, 256])
解读: 4个位置，每个位置是256维向量


In [11]:
# 关键一步：相加！
# token_embeddings: (3, 4, 256)
# pos_embeddings:         (4, 256)
# PyTorch会自动广播：把(4,256)加到每个样本上
input_embeddings = token_embeddings + pos_embeddings

print(f"词嵌入形状:   {token_embeddings.shape}")
print(f"位置嵌入形状: {pos_embeddings.shape}")
print(f"最终输入形状: {input_embeddings.shape}   ← 形状不变，但内容变了")

词嵌入形状:   torch.Size([3, 4, 256])
位置嵌入形状: torch.Size([4, 256])
最终输入形状: torch.Size([3, 4, 256])   ← 形状不变，但内容变了


## 完整数据流总结

## 各阶段形状变化一览

| 阶段 | 数据形式 | 形状 | 示例 |
|:---|:---|:---|:---|
| 原始文本 | 字符串 | — | `"I HAD always..."` |
| 分词后 | 字符串列表 | `(seq_len,)` | `['I', 'HAD', 'always', ...]` |
| 编码后 | 整数序列 | `(seq_len,)` | `[2, 1, 5, ...]` |
| 数据采样 | 整数张量对 | `(N, ctx_len)` | `inputs: (3, 4)` |
| 词嵌入后 | 浮点张量 | `(N, ctx_len, dim)` | `(3, 4, 256)` |
| 加位置编码 | 浮点张量 | `(N, ctx_len, dim)` | `(3, 4, 256)` |

> `N` = 样本数, `ctx_len` = 上下文长度, `dim` = 嵌入维度

这就是数据准备的完整主线。理解了这些形状变化，你就掌握了训练数据的核心逻辑！